## Preprocessing of Midi Files

Goals:
- read MIDI files
- align with CSV file (train, test, validation split)
- filter non-4/4
- filter fills
- extract Information: 
    - onsets (hits) (binary)
    - continuous derivation (-.5, .5)
    - velocity (0 - 127 --> 0 - 1)

TODOS
- create grid based on rounded onset times and instrument group
- create rhythmic offset matrix for training purposes


In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import pretty_midi

In [2]:
# Constants

# DS_ROOT = "e-gmd-v1.0.0"
DS_ROOT = "../../e-gmd-v1.0.0" # this assumes the dataset to be in a solder next to the git repository
N_INSTRUMENTS = 9

In [3]:
def pitch_to_category(pitches):
    """
    Maps the Midi pitch values 0 - 127 to the Drum category according to the Paper's Appendix B
    """

    mapping = {
        36 : 36, # Kick
        38 : 38, # Snare Head
        40 : 38, # Snare Rim
        37 : 38, # Snare Side Stick
        48 : 50, # Tom High
        50 : 50,
        45 : 47,
        47 : 47,
        43 : 43,
        58 : 43,
        46 : 46,
        26 : 46,
        42 : 42,
        22 : 42,
        44 : 42,
        49 : 49,
        55 : 49,
        57 : 49,
        52 : 49,
        51 : 51,
        59 : 51,
        53 : 51,

    }

    # how many instruments?
    n_instruments = len(set(mapping.values()))
    if n_instruments != N_INSTRUMENTS:
        raise ValueError("Global constant N_INSTRUMENTS does not fit unique values in mapping dict")
    
    
    categories = []
    for pitch in pitches:
        # write 0 if value is missing in mapping
        categories.append(mapping.get(pitch, 0))

    return np.array(categories)


In [4]:
# Define variable-length 16th grid

def generate_onset_grid(tempo_bpm, onsets):
    
    n_instruments = 8 # TODO look up in papers

    bps = tempo_bpm / 60. # beats per second
    sixteenth_ps = bps * 4 # sixteenth notes per second

    last_onset = onsets[-1]
    n_onsets = int(np.ceil(last_onset*sixteenth_ps))
    grid = np.zeros((n_onsets, n_instruments)) 


    # round to closest;
    # save the distances 
    # norm to grid instead of milliseconds
    step = 1/sixteenth_ps
    nearest = np.round(onsets / step) * step

    distance = np.abs(onsets - nearest)

    return grid

In [ ]:
# Read MIDI file, extract all needed data

def read_midi(midi_path : Path, tempo_bpm):

    try:
        midi_data = pretty_midi.PrettyMIDI(midi_path)
    except Exception as e:
        raise ValueError(f"Failed to parse MIDI file {midi_path}: {e}")
    
    # Get drum track (should be only drum track)
    drum_track = None
    for instrument in midi_data.instruments:
        if instrument.is_drum:
            drum_track = instrument
            break

    if drum_track is None:
        raise ValueError(f"No drum track found in {midi_path}")
    
    # iterate over notes
    onsets = []
    pitches = []
    velocities = []
    for note in drum_track.notes:
        onsets.append(note.start)
        pitches.append(note.pitch)
        velocities.append(note.velocity)


    # sort by onset time
    sort_idx = np.argsort(onsets)
    onsets = np.array(onsets)[sort_idx]
    pitches = np.array(pitches)[sort_idx]
    velocities = np.array(velocities)[sort_idx]


    # calc 1/16th notes per second
    bps = tempo_bpm / 60. # beats per second
    sixteenth_ps = bps * 4 # sixteenth notes per second

    # get vector / grid length for binary Matrix
    last_onset = onsets[-1] # in sec
    n_grid_onsets = int(np.ceil(last_onset*sixteenth_ps))

    # snap to grid
    step = 1/sixteenth_ps
    nearest = np.round(onsets / step) * step # snapped values
    nearest_idc = np.rint(nearest * sixteenth_ps).astype(int) # get snapped onsets as grid indices
    
    offsets = np.abs(onsets - nearest) # timing derivations ('feel')
    offsets_idc = offsets * sixteenth_ps # relative to grid, not in seconds

    # split into one matrix
    
    

    # create instrument grouped array --> "round" different drum parts (ride bell, ride edge --> ride)
    grouped_pitches = pitch_to_category(pitches)
    pitch_categories = np.array([36, 38, 50, 47, 43, 46, 42, 49, 51])
    pitch_to_row = {p: i for i, p in enumerate(pitch_categories)}
    pitch_rows = np.array([pitch_to_row[p] for p in grouped_pitches])
    
    # create grid
    grid = np.zeros((len(pitch_categories), n_grid_onsets), dtype=np.uint8)
    



    # onset_grid = generate_onset_grid(tempo_bpm=tempo_bpm, onsets=onsets)
    
    # print(onset_grid.shape)
    
    return drum_track


In [ ]:
# Iterate over csv
# read all MIDI files

csv_path = Path(DS_ROOT) / "e-gmd-v1.0.0.csv"
df = pd.read_csv(csv_path)

for i, row in df.iterrows():

    # early stopping for testing
    if i == 2:
        break

    # get BPM from csv
    bpm = row['bpm']

    # load file
    filepath =  Path(DS_ROOT) / row["midi_filename"]
    print(filepath)

    # call reader function
    test = read_midi(filepath, bpm)


../../e-gmd-v1.0.0/drummer1/eval_session/1_funk-groove1_138_beat_4-4_1.midi
[7 8 0 0 8 6 8 1 6 8 1 6 8 1 6 0 8 6 8 1 6 8 1 6 8 0 6 0 8 6 1 8 6 8 1 6 8
 1 0 6 8 0 1 6 8 6 8 1 1 1 6 0 8 1 6 0 8 6 1 8 6 8 1 6 8 1 6 0 8 6 8 6 8 1
 6 8 1 6 8 0 0 8 1 6 6 8 1 8 6 1 1 6 7 0 6 8 6 1 8 6 8 0 1 6 8 0 6 8 1 6 8
 1 6 8 1 6 0 8 0 6 1 8 6 8 1 1 1 6 1 8 0 6 8 0 6 1 8 6 8 1 6 8 1 6 0 8 0 6
 1 8 6 8 1 1 1 6 0 8 6 0 8 6 1 8 6 8 1 6 8 1 6 0 8 6 8 6 1 8 6 8 1 6 8 0 8
 0 6 1 8 6 8 1 6 8 1 6 0 7 1 6 8 6 1 8 6 0 8 1 6 0 8 6 1 8 6 8 1 6 8 1 0 6
 8 6 8 1 6 8 1 1 1 6 0 8 6 8 0 6 8 1 6 8 1 6 8 1 6 0 8 0 6 1 8 6 8 1 1 1 1
 6 0 8 6 8 0 6 1 8 6 8 1 6 8 1 6 0 8 6 8 6 1 8 6 8 1 6 8 0 8 0 1 8 6 6 8 1
 6 8 1 6 4 1 6 8 0 6 8 1 6 0 8 1 6 8 0 6 1 8 6 8 1 6 8 1 6 0 8 0 6 8 1 8 1
 1 1 1 6 8 0 8 0 6 1 8 6 8 1 6 8 1 6 0 8 0 6 8 1 6 8 1 1 1 1 6 0 8 0 8 6 1
 8 6 8 1 6 8 1 6 0 8 6 8 6 8 1 6 8 1 6 0 8 0 8 6 1 8 6 8 1 6 8 1 6 7 0 6 8
 1 8 6]
../../e-gmd-v1.0.0/drummer1/eval_session/1_funk-groove1_138_beat_4-4_10.midi


## Questions

- do i concatenate the 2-bar snippets (sliding window of 1 bar) fully, step-by-step or not at all?